# LQ-1 Proof Repair Experiment

Adapted for the **`shannon-llm-integration`** branch. Differences from the
original notebook are called out below, because several are load-bearing.

This notebook configures the experiment harness for the LQ-1 controlled
proof-repair benchmark.

## Benchmark

The benchmark uses the `sampling_bound` lemma from LQ-1. The theorem
statement and assumptions are preserved, while the final `smt()` tactic is
removed to create an incomplete-proof case.

- Spec: `lq1-broken-repair`
- Corpus: `LQ1Corpus`
- Benchmark type: controlled synthetic repair
- Failure category: incomplete proof
- Trials: one proof case

In [1]:
import logging
import os
import sys
from pathlib import Path

# Project root on the path, and cwd there (corpus paths are root-relative).
PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == "notebooks" else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

# .env is not auto-loaded anywhere in this repo (no dotenv dependency).
env_path = PROJECT_ROOT / ".env"
if env_path.is_file():
    for raw in env_path.read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
# The embeddings endpoint is called thousands of times per trial; its request
# log drowns everything else in this notebook.
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("openai").setLevel(logging.WARNING)

print(f"Project root      : {PROJECT_ROOT}")
print(f"DEEPSEEK_API_KEY  : {'set' if os.environ.get('DEEPSEEK_API_KEY') else 'NOT SET'}")
print(f"ANTHROPIC_API_KEY : {'set' if os.environ.get('ANTHROPIC_API_KEY') else 'NOT SET'}")

Project root      : /Users/Aney/AI4EC
DEEPSEEK_API_KEY  : NOT SET
ANTHROPIC_API_KEY : NOT SET


## Preflight

Fails fast on the two things that otherwise break a run *after* it has started spending.

In [2]:
from integration.agent.config import AgentConfig
from integration.agent.ec_version import detect_target_version
from integration.experiment.__main__ import _embeddings_endpoint_status

_probe = AgentConfig()

ok, detail = _embeddings_endpoint_status(_probe)
print(f"Embeddings endpoint : {'OK' if ok else 'UNAVAILABLE'} — {detail}")
if not ok:
    print("  -> start LM Studio and load an embedding model; every provider needs it.")

print(f"EasyCrypt binary    : {'found' if _probe.easycrypt_bin.exists() else 'MISSING'} ({_probe.easycrypt_bin})")
_target = detect_target_version(_probe.easycrypt_bin)
print(f"EasyCrypt version   : {_target.version} (via {_target.method}, confidence {_target.confidence})")

Embeddings endpoint : UNAVAILABLE — APIConnectionError: Connection error.
  -> start LM Studio and load an embedding model; every provider needs it.
EasyCrypt binary    : MISSING (/Users/Aney/AI4EC/integration/extern/easycrypt/_build/default/src/ec.exe)
EasyCrypt version   : None (via binary_missing, confidence none)


## Configuration

In [ ]:
# --- Spec -------------------------------------------------------------------
SPEC_NAME = "lq1-broken-repair"  

# --- Provider ---------------------------------------------------------------
PROVIDER = "deepseek"                    # deepseek | anthropic | lm_studio

# deepseek: deepseek-v4-flash (cheap) | deepseek-v4-pro
# anthropic: claude-opus-5 (default) | claude-sonnet-5
# lm_studio: whatever is loaded locally
MODEL = "deepseek-v4-flash"

THINKING_MODE = "adaptive"               # disabled | enabled | adaptive
# Keep this on. Matched on the same three proofs, same model, adaptive
# accepted 43% of productive calls against 4% for disabled -- and all
# three thinking-disabled trials went STUCK having accepted ~nothing.
REASONING_EFFORT = None                  # deepseek: high|max ; anthropic: low..max
EMBED_MODEL = "text-embedding-nomic-embed-text-v1.5"

# --- Budget -----------------------------------------------------------------
MAX_TRIALS = 1                           # the LQ-1 corpus currently contains one benchmark case
ADAPTIVE_MULTIPLIER = 1.4                # step budget = 1.4x tactic lines
MIN_STEPS = 10                           # floor for the step budget
STUCK_LIMIT = 20
TOP_K_PREMISES = 10
LLM_MAX_TOKENS = 32768                   # mean output was ~13.2k against the
                                         # old 16384 cap (82% of ceiling), so
                                         # calls truncated routinely. Truncation
                                         # is a budget problem, not a reason to
                                         # turn thinking off.
COST_LIMIT_USD = 1.00                    # enforced DURING the run; None = uncapped

DATA_DIR = Path("data")
OUTPUT_DIR = None                        # None = auto-timestamped

In [4]:
from integration.agent.config import (
    LLM_PROVIDER_ANTHROPIC,
    LLM_PROVIDER_DEEPSEEK,
    PAID_LLM_PROVIDERS,
    apply_anthropic_provider,
    apply_deepseek_provider,
)
from integration.experiment.config import ExperimentConfig

agent = AgentConfig(
    top_k=TOP_K_PREMISES,
    llm_max_tokens=LLM_MAX_TOKENS,
    embed_model=EMBED_MODEL,
)

if PROVIDER == LLM_PROVIDER_DEEPSEEK:
    apply_deepseek_provider(agent, model=MODEL, thinking=THINKING_MODE,
                            reasoning_effort=REASONING_EFFORT)
elif PROVIDER == LLM_PROVIDER_ANTHROPIC:
    apply_anthropic_provider(agent, model=MODEL, thinking=THINKING_MODE,
                             reasoning_effort=REASONING_EFFORT)
else:
    agent.llm_model = MODEL  # local: free, no cap needed

exp_config = ExperimentConfig(
    spec_name=SPEC_NAME,
    trials=MAX_TRIALS,
    stuck_limit=STUCK_LIMIT,
    data_dir=DATA_DIR,
    agent=agent,
    sort_by_difficulty=True,
    adaptive_steps_multiplier=ADAPTIVE_MULTIPLIER,
    min_adaptive_steps=MIN_STEPS,
    # A cap is meaningless for a local model and is refused for a model with
    # no published rates, so only set it where it can actually be enforced.
    cost_limit_usd=COST_LIMIT_USD if PROVIDER in PAID_LLM_PROVIDERS else None,
)
if OUTPUT_DIR is not None:
    exp_config.output_dir = Path(OUTPUT_DIR)

exp_config = exp_config.with_agent_defaults()   # builds the SpendBudget

print(f"Spec            : {exp_config.spec_name}")
print(f"Provider/model  : {agent.llm_provider} / {agent.llm_model}")
print(f"Thinking/effort : {agent.llm_thinking} / {agent.llm_reasoning_effort}")
print(f"Output dir      : {exp_config.output_dir}")
print(f"Adaptive steps  : {ADAPTIVE_MULTIPLIER}x (min {MIN_STEPS})")
print(f"Spend cap       : {exp_config.agent.spend_budget.status() if exp_config.agent.spend_budget else 'none (uncapped)'}")

Spec            : lq1-broken-repair
Provider/model  : deepseek / deepseek-v4-flash
Thinking/effort : adaptive / None
Output dir      : /Users/Aney/AI4EC/integration/output/experiments/run-20260806T191625Z
Adaptive steps  : 1.4x (min 10)
Spend cap       : $0.0000 of $1.00 spent (0 calls, $1.0000 remaining)


## Preview: proof cases, shortest first

In [5]:
from integration.experiment.__main__ import _build_spec, _with_sandbox_dir

preview_spec = _with_sandbox_dir(
    _build_spec(SPEC_NAME, DATA_DIR),
    DATA_DIR,
    exp_config.output_dir / "sandboxes",
)

corpus = preview_spec.corpus
all_cases = sorted(
    corpus.load_cases(),
    key=lambda case: len(case.tactic_lines),
)

print(
    f"Available proofs: {len(all_cases)}"
    f"   |   attempting: {min(MAX_TRIALS, len(all_cases))}\n"
)

print(f"{'#':<3} {'Name':<20} {'Lines':>6} {'Steps':>7}")
print("-" * 40)

for i, case in enumerate(all_cases):
    n = len(case.tactic_lines)
    marker = " <--" if i < MAX_TRIALS else ""
    print(
        f"{i:<3} {case.name:<20} "
        f"{n:>6} {exp_config.steps_for_case(n):>7}{marker}"
    )

Available proofs: 1   |   attempting: 1

#   Name                  Lines   Steps
----------------------------------------
0   sampling_bound            5      10 <--


## Confirm paid usage

`run_experiment` does **not** gate paid providers — the confirmation lives in
the CLI entry points only, so calling it directly from a notebook would spend
real money with no prompt. This cell reproduces the CLI's gate.

`sys.stdin.readline()` does not work under ipykernel, so this uses `input()`,
which Jupyter routes to the frontend prompt.


In [ ]:
from integration.agent.config import PAID_LLM_PROVIDERS
from integration.experiment.paid_confirm import (
    CONFIRMATION_PHRASE,
    format_paid_provider_warning,
)

CONFIRMED = agent.llm_provider not in PAID_LLM_PROVIDERS
if CONFIRMED:
    print(f"{agent.llm_provider}: not a paid provider, no confirmation needed.")
else:
    print(format_paid_provider_warning(
        config=agent,
        trials=MAX_TRIALS,
        informal=False,
    ))
    CONFIRMED = input().strip() == CONFIRMATION_PHRASE
    print("Confirmed." if CONFIRMED else "NOT confirmed - the run cell will refuse.")


## Run

Shortest-first, stopping as soon as `COST_LIMIT_USD` is reached.

> This run attempts to repair the incomplete `sampling_bound` proof. The
> theorem statement and assumptions are unchanged; only the final
> proof-closing `smt()` tactic was removed.

In [ ]:
from integration.experiment.__main__ import _build_spec, _with_sandbox_dir
from integration.experiment.runner import run_experiment

exp_config.output_dir.mkdir(parents=True, exist_ok=True)
spec = _with_sandbox_dir(
    _build_spec(exp_config.spec_name, DATA_DIR),
    DATA_DIR,
    exp_config.output_dir / "sandboxes",
)

if not CONFIRMED:
    raise RuntimeError(
        "Paid usage was not confirmed - run the confirmation cell above. "
        "No API call was made."
    )

result = run_experiment(spec, exp_config)

print(f"\nSpec      : {result.spec_name}   mode: {result.mode}")
print(f"Trials    : {result.trials_run} run, {result.trials_skipped} skipped")
print(f"Outcomes  : {result.successes} complete, {result.stuck} stuck, {result.max_steps} max-steps, {result.errors} errors")
if result.estimated_cost:
    print(f"Cost      : ${result.estimated_cost['usd']:.6f}")
if result.budget:
    print(f"Budget    : {result.budget['spent_usd']:.6f} / {result.budget['limit_usd']:.2f} USD"
          f"{'  (STOPPED EARLY)' if result.budget_stopped else ''}")
print(f"Output    : {result.output_dir}")

## Results

The primary outcome is whether the model reconstructs a complete proof for
the broken `sampling_bound` case within the configured step and cost limits.

In [ ]:
print(f"{'#':<3} {'Name':<20} {'Outcome':<10} {'Steps':>5} {'Calls':>6} {'Cost':>10}  Route")
print("-" * 78)
model_repairs = 0

for t in result.trial_results:
    calls = t.token_usage.calls
    cost = (t.estimated_cost or {}).get("usd", 0.0)

    if t.reason == "COMPLETE":
        route = "MODEL REPAIR"
        model_repairs += 1
    else:
        route = "model, unsolved" if calls else "—"

    print(
        f"{t.trial_id:<3} {t.name:<20} {t.reason:<10} "
        f"{t.steps:>5} {calls:>6} {cost:>10.6f}  {route}"
    )

attempted = len(result.trial_results)

print(f"\nRepaired by the model: {model_repairs} of {attempted}")

#   Name                 Outcome    Steps  Calls       Cost  Route
------------------------------------------------------------------------------


NameError: name 'result' is not defined

## Per-failure diagnostics

Which failures were proof-level vs import-level, and therefore which kind of
changelog evidence the retrieval routed to (`ec_errors.py` wiring).

In [ ]:
from collections import Counter

from integration.agent.ec_errors import classify_error

kinds, accepted_total, refreshes = Counter(), 0, 0
for trial_dir in sorted((result.output_dir / "trials").iterdir()):
    log = trial_dir / "agent_log.json"
    if not log.is_file():
        continue
    events = json.loads(log.read_text()).get("events", [])
    refreshes += sum(1 for e in events if e.get("event") == "changelog_hint_refresh")
    for e in events:
        if e.get("event") != "iteration" or e.get("action") != "tactic":
            continue
        if e.get("outcome") in ("accepted", "complete"):
            accepted_total += 1
        elif e.get("outcome") == "failed" and e.get("error"):
            kinds[classify_error(e["error"]).kind] += 1

print(f"Tactics accepted by the model : {accepted_total}")
print(f"Live changelog hint refreshes : {refreshes}")
print("\nFailure kinds (drives which evidence is retrieved):")
for kind, n in kinds.most_common():
    route = "tactic changelog entries" if kind in {"tactic_error", "proof_incomplete"} else "import evidence"
    print(f"  {kind:<18} {n:>3}   -> {route}")

## Inspect one trial

Set `TRIAL` to a trial that used the model.

In [ ]:
TRIAL = next((t.trial_id for t in result.trial_results if t.token_usage.calls), 0)
trial_dir = result.output_dir / "trials" / f"trial_{TRIAL:03d}"
print(f"=== {trial_dir.name} ===\n")

boot = trial_dir / "bootstrap_result.json"
if boot.is_file():
    b = json.loads(boot.read_text())
    print(f"Replayed {b['accepted_count']}/{b['total_count']} original tactics")
    if b.get("failed_tactic"):
        print(f"First tactic that no longer applies:\n  {b['failed_tactic'][:300]}\n")

hints = trial_dir / "changelog_hints.txt"
if hints.is_file():
    text = hints.read_text()
    print(f"--- hint block shown to the model ({len(text)} chars) ---")
    print(text[:1500])

log = trial_dir / "agent_log.json"
if log.is_file():
    print("\n--- what the model tried ---")
    for e in json.loads(log.read_text()).get("events", []):
        if e.get("event") == "iteration" and e.get("action") == "tactic":
            print(f"  {e.get('outcome'):<9} {(e.get('tactic') or '')[:70]}")

## Comparing providers

Re-run with `PROVIDER = "anthropic"` / `MODEL = "claude-opus-5"` and compare.
`sort_by_difficulty=True` makes the case order deterministic, so runs are
directly comparable without a shared seed.

The three numbers worth comparing:

1. **Repaired by the model** — not `successes`, which includes free replays.
2. **`hint_uptake.rate_among_scorable`** — only meaningful once a run actually
   lands tactics; `None` means not measurable.
3. **`estimated_cost.usd` per repair** — a model that repairs twice as much for
   ten times the cost is a different trade, not a better one.

In [ ]:
summary_path = result.output_dir / "summary.json"
print(f"Full summary: {summary_path}")
print(json.dumps({
    "spec": result.spec_name,
    "provider": agent.llm_provider,
    "model": agent.llm_model,
    "thinking": agent.llm_thinking,
    "effort": agent.llm_reasoning_effort,
    "repaired_by_model": model_repairs,
    "attempted_by_model": attempted,
    "zero_llm_replays": zero_llm,
    "cost_usd": (result.estimated_cost or {}).get("usd"),
    "hint_uptake": result.repair_metrics.get("hint_uptake"),
}, indent=2))